# Historical exploratory notebook

This notebook preserves the original exploratory code. Saved outputs and execution metadata have been removed for the source candidate. No cells were rerun. Read the [methodology corrections](../docs/DISSERTATION_NOTES.md) before interpreting or rerunning it. It requires separately verified private input data; running it against a later dataset is not reproduction of the April results.


In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import os

# --------------------------------------------------------
# PLOTTING STYLE
# --------------------------------------------------------
sns.set_theme(style="whitegrid", context="paper", font_scale=1.2)
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['savefig.dpi'] = 300

# --------------------------------------------------------
# PATHS
# Notebook lives in notebooks/ — data in data/processed/
# --------------------------------------------------------
DATA_PATH  = "../data/processed/master_dataset.csv"
OUTPUT_DIR = "../reports/figures/eda"

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Output directory: {os.path.abspath(OUTPUT_DIR)}")


In [ ]:
# --------------------------------------------------------
# LOAD DATA
# --------------------------------------------------------
df = pd.read_csv(DATA_PATH)

TARGET_COL = 't2d_label'
ID_COLS    = ['respondent_id', 'survey_cycle', 'interview_weight']
LEAKAGE    = ['doctor_told_diabetes', 'taking_insulin',
              'taking_diabetes_pills', 'hba1c_percent']

# --------------------------------------------------------
# 16 ACTIVE FEATURES (aligned with config.py after comments)
# --------------------------------------------------------
NUMERICAL = [
    'age_years', 'poverty_ratio', 'bmi', 'waist_circumference',
    'waist_to_height_ratio', 'systolic_bp', 'diastolic_bp', 'pulse_60s'
]
NOMINAL = [
    'gender_code', 'ethnicity_code', 'history_high_bp',
    'smoked_100_cigs', 'family_history_diabetes', 'vigorous_activity'
]
ORDINAL  = ['education_level', 'alcohol_frequency']
ALL_16   = NUMERICAL + NOMINAL + ORDINAL

# Human-readable labels for axis annotations
LABELS = {
    'age_years':              'Age (years)',
    'poverty_ratio':          'Poverty Ratio',
    'bmi':                    'BMI',
    'waist_circumference':    'Waist Circumference (cm)',
    'waist_to_height_ratio':  'Waist-to-Height Ratio',
    'systolic_bp':            'Systolic BP (mmHg)',
    'diastolic_bp':           'Diastolic BP (mmHg)',
    'pulse_60s':              'Resting Pulse (bpm)',
    'gender_code':            'Gender',
    'ethnicity_code':         'Ethnicity',
    'history_high_bp':        'High BP History',
    'smoked_100_cigs':        'Smoked 100+ Cigarettes',
    'family_history_diabetes':'Family Diabetes History',
    'vigorous_activity':      'Vigorous Activity',
    'education_level':        'Education Level',
    'alcohol_frequency':      'Alcohol Frequency (days/year)',
}

df[TARGET_COL] = pd.to_numeric(df[TARGET_COL], errors='coerce')
df_model = df[ALL_16 + [TARGET_COL]].copy()

print(f"Full dataset shape:  {df.shape}")
print(f"Model dataset shape: {df_model.shape}")
print(f"Class distribution:")
counts = df_model[TARGET_COL].value_counts()
print(f"  Non-diabetic (0): {counts.get(0,0):,}  ({counts.get(0,0)/len(df_model)*100:.1f}%)")
print(f"  Diabetic     (1): {counts.get(1,0):,}  ({counts.get(1,0)/len(df_model)*100:.1f}%)")


In [ ]:
# --------------------------------------------------------
# FIGURE D.1.1 — CLASS DISTRIBUTION
# --------------------------------------------------------
df_plot = df_model.dropna(subset=[TARGET_COL]).copy()
df_plot[TARGET_COL] = df_plot[TARGET_COL].astype(int)

counts  = df_plot[TARGET_COL].value_counts().sort_index()
labels  = ['Non-Diabetic', 'Diabetic']
colours = ['#4A90D9', '#E05A4E']
total   = counts.sum()

fig, ax = plt.subplots(figsize=(7, 5))
bars = ax.bar(labels, counts.values, color=colours, width=0.5, edgecolor='white')

for bar, count in zip(bars, counts.values):
    pct = count / total * 100
    ax.text(bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 80,
            f'{count:,}\n({pct:.1f}%)',
            ha='center', va='bottom', fontsize=11, fontweight='bold')

ax.set_title('Class Distribution — Diabetes Label', fontsize=14, fontweight='bold', pad=12)
ax.set_ylabel('Number of Records', fontsize=12)
ax.set_ylim(0, counts.max() * 1.18)
ax.axhline(y=0, color='black', linewidth=0.8)
sns.despine()
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'D1_1_class_distribution.png'))
plt.show()
print(f"Saved D1_1_class_distribution.png")


In [ ]:
# --------------------------------------------------------
# FIGURE 31 — FEATURE MISSINGNESS PROFILE
# Shows all candidate variables: 16 active features and
# 10 excluded variables. Each excluded variable label
# states the specific reason for exclusion. A single
# orange colour is used for all excluded variables to
# distinguish them from the active set (blue).
# --------------------------------------------------------

EXCLUDED_VARS = {
    'weight_kg':             'Weight (kg) [excluded: algebraic component of BMI]',
    'height_cm':             'Height (cm) [excluded: algebraic component of WHtR]',
    'arm_circumference_cm':  'Arm Circumference (cm) [excluded: redundant with waist/BMI]',
    'arm_length_upper_cm':   'Arm Length (cm) [excluded: low correlation with target]',
    'leg_length_upper_cm':   'Upper Leg Length (cm) [excluded: low relevance to T2D]',
    'highest_weight_ever_kg':'Highest Weight Ever (kg) [excluded: self-report bias]',
    'diet_energy_kcal':      'Total Energy Intake (kcal) [excluded: self-report bias]',
    'diet_sodium_mg':        'Sodium Intake (mg) [excluded: self-report bias]',
    'diet_carbs_gm':         'Carbohydrate Intake (g) [excluded: self-report bias]',
    'diet_caffeine_mg':      'Caffeine Intake (mg) [excluded: self-report bias]',
}

# Build combined variable list from what exists in the dataset
all_labels_combined = {**LABELS}
all_vars_combined = ALL_16.copy()

for var, label in EXCLUDED_VARS.items():
    if var in df.columns:
        all_vars_combined.append(var)
        all_labels_combined[var] = label

miss = (df[all_vars_combined].isna().mean() * 100).round(2)
miss_df = pd.DataFrame({
    'Feature': [all_labels_combined.get(c, c) for c in all_vars_combined],
    'Missing (%)': [miss[c] for c in all_vars_combined],
    'Status': ['Active' if c in ALL_16 else 'Excluded' for c in all_vars_combined]
}).sort_values('Missing (%)', ascending=True)

fig, ax = plt.subplots(figsize=(12, 10))

colours_bar = ['#4A90D9' if s == 'Active' else '#D4834E' for s in miss_df['Status']]
bars = ax.barh(miss_df['Feature'], miss_df['Missing (%)'], color=colours_bar, edgecolor='white')

ax.axvline(x=30, color='#C0392B', linestyle='--', linewidth=1.2)

for bar, val in zip(bars, miss_df['Missing (%)']):
    if val > 0.5:
        ax.text(val + 0.3, bar.get_y() + bar.get_height()/2,
                f'{val:.1f}%', va='center', fontsize=8)

from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='#4A90D9', label='Active (16 features)'),
    Patch(facecolor='#D4834E', label='Excluded (reason in label)'),
    plt.Line2D([0], [0], color='#C0392B', linestyle='--',
               label='30% Missingness Threshold')
]
ax.legend(handles=legend_elements, fontsize=9, loc='lower right')

ax.set_xlabel('Missing Values (%)', fontsize=12)
ax.set_title('Feature Missingness Profile: Active and Excluded Variables',
             fontsize=13, fontweight='bold', pad=12)
sns.despine()
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'D1_2_missingness.png'), bbox_inches='tight')
plt.show()

active_count = sum(1 for s in miss_df['Status'] if s == 'Active')
excluded_count = sum(1 for s in miss_df['Status'] if s == 'Excluded')
print(f"Active features: {active_count}")
print(f"Excluded variables: {excluded_count}")
print(f"Total shown: {active_count + excluded_count}")
print(f"\nNote: Dietary variables are at ~9% missingness, well below the 30% threshold.")
print(f"They were excluded for self-reporting bias, not missingness.")
print("\nSaved D1_2_missingness.png")


In [ ]:
# --------------------------------------------------------
# FIGURE D.1.3 — CORRELATION HEATMAP (16 features + target)
# --------------------------------------------------------
corr_df = df_model.copy()
corr_df['gender_code'] = corr_df['gender_code'].map({1: 0, 2: 1})  # 0=Male, 1=Female

corr = corr_df[ALL_16 + [TARGET_COL]].corr()

readable_cols = [LABELS.get(c, c) for c in ALL_16] + ['T2D Label']
corr.index   = readable_cols
corr.columns = readable_cols

fig, ax = plt.subplots(figsize=(16, 14))
mask = np.zeros_like(corr, dtype=bool)
mask[np.triu_indices_from(mask, k=1)] = True  # upper triangle masked

sns.heatmap(
    corr, mask=mask, annot=True, fmt='.2f',
    cmap='RdBu_r', center=0, linewidths=0.4,
    annot_kws={'size': 8}, cbar_kws={'shrink': 0.8}, ax=ax
)
ax.set_title('Correlation Matrix — 16 Active Features + Target', fontsize=14, fontweight='bold', pad=14)
plt.xticks(rotation=45, ha='right', fontsize=9)
plt.yticks(fontsize=9)
plt.subplots_adjust(bottom=0.28, left=0.18)
plt.savefig(os.path.join(OUTPUT_DIR, 'D1_3_correlation_heatmap.png'), bbox_inches='tight')
plt.show()

print("\nTop correlations with T2D label:")
target_corr = corr['T2D Label'].drop('T2D Label').sort_values(key=abs, ascending=False)
print(target_corr.head(10).to_string())
print("\nSaved D1_3_correlation_heatmap.png")


In [ ]:
# --------------------------------------------------------
# FIGURE D.1.4 — KEY NUMERICAL FEATURES BY DIABETES STATUS
# Age, BMI, Waist Circumference, Waist-to-Height Ratio
# (top 4 numerical features by SHAP importance)
# --------------------------------------------------------
plot_features = ['age_years', 'bmi', 'waist_circumference', 'waist_to_height_ratio']
plot_titles   = [LABELS[f] for f in plot_features]

df_plot = df_model.dropna(subset=[TARGET_COL]).copy()
df_plot[TARGET_COL] = df_plot[TARGET_COL].astype(int)

COLOURS = {0: '#4A90D9', 1: '#E05A4E'}
STATUS  = {0: 'Non-Diabetic', 1: 'Diabetic'}

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for i, (col, title) in enumerate(zip(plot_features, plot_titles)):
    ax = axes[i]
    data_clean = df_plot.dropna(subset=[col])
    sns.violinplot(
        data=data_clean, x=TARGET_COL, y=col,
        palette=COLOURS, hue=TARGET_COL,
        inner='quartile', legend=False, ax=ax
    )
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_xlabel('Diabetes Status', fontsize=10)
    ax.set_ylabel(title, fontsize=10)
    ax.set_xticks([0, 1])
    ax.set_xticklabels(['Non-Diabetic', 'Diabetic'], fontsize=9)

handles = [mpatches.Patch(color=COLOURS[k], label=STATUS[k]) for k in [0, 1]]
fig.suptitle('Key Feature Distributions by Diabetes Status', fontsize=14, fontweight='bold', y=1.02)
handles = [mpatches.Patch(color=COLOURS[k], label=STATUS[k]) for k in [0, 1]]
fig.legend(handles=handles, loc='lower center', ncol=2, fontsize=11,
           frameon=False, bbox_to_anchor=(0.5, -0.04))
plt.tight_layout(rect=[0, 0.04, 1, 0.98])
plt.savefig(os.path.join(OUTPUT_DIR, 'D1_4_feature_distributions.png'), bbox_inches='tight')
plt.show()
print("Saved D1_4_feature_distributions.png")


In [ ]:
# --------------------------------------------------------
# SUMMARY — FILES SAVED
# --------------------------------------------------------
import glob
saved = sorted(glob.glob(os.path.join(OUTPUT_DIR, '*.png')))
print(f"EDA figures saved to: {os.path.abspath(OUTPUT_DIR)}")
print()
for f in saved:
    size_kb = os.path.getsize(f) / 1024
    print(f"  {os.path.basename(f):<45} {size_kb:.0f} KB")
